# Lab 5: Neural Networks, Convolutions, and Transfer Learning

This notebook follows the seven parts of the Week 5 handout. Run it in Kaggle with the **Intel Image Classification** dataset (`puneet6060`), Internet enabled, and a T4 GPU. The currently selected VS Code kernel is CPU-only and has no dataset mounted; Part 2 and the written answers can still be checked here, but image training and test results must be run in Kaggle. No accuracy, class confusion, or visual observations are filled in as if they had been measured.

The notebook saves the trained state dictionary under `/kaggle/working` in Kaggle. After running it there, download the model and use the real outputs in your README and post.

In [2]:
import copy
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("/kaggle/input/intel-image-classification")
TRAIN_DIR = ROOT / "seg_train" / "seg_train"
TEST_DIR = ROOT / "seg_test" / "seg_test"
DATA_AVAILABLE = TRAIN_DIR.is_dir() and TEST_DIR.is_dir()
classes = (
    sorted(path.name for path in TRAIN_DIR.iterdir() if path.is_dir())
    if TRAIN_DIR.is_dir()
    else []
)
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Device:", device)
if device.type != "cuda":
    print("Enable a Kaggle T4 GPU before training; this runtime is CPU-only.")
print("Dataset root:", ROOT)
print("Dataset available:", DATA_AVAILABLE)
if not DATA_AVAILABLE:
    print("Add the Intel Image Classification dataset (puneet6060) to the Kaggle notebook.")
else:
    for class_name in classes:
        train_count = sum(
            path.suffix.lower() in IMAGE_SUFFIXES
            for path in (TRAIN_DIR / class_name).iterdir()
            if path.is_file()
        )
        test_count = sum(
            path.suffix.lower() in IMAGE_SUFFIXES
            for path in (TEST_DIR / class_name).iterdir()
            if path.is_file()
        )
        print(f"{class_name:10s} train {train_count:5d}   test {test_count:4d}")

Device: cpu
Enable a Kaggle T4 GPU before training; this runtime is CPU-only.
Dataset root: \kaggle\input\intel-image-classification
Dataset available: False
Add the Intel Image Classification dataset (puneet6060) to the Kaggle notebook.


In [3]:
if DATA_AVAILABLE:
    fig, axes = plt.subplots(2, len(classes), figsize=(15, 5))
    sample_paths = []
    for column, class_name in enumerate(classes):
        image_paths = sorted(
            path for path in (TRAIN_DIR / class_name).iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        )[:2]
        sample_paths.extend(image_paths[:1])
        for row_index in range(2):
            axis = axes[row_index, column]
            if row_index < len(image_paths):
                with Image.open(image_paths[row_index]) as source_image:
                    axis.imshow(source_image.convert("RGB"))
                axis.set_title(class_name)
            axis.axis("off")
    plt.tight_layout()
    plt.show()

    sample_image = np.asarray(Image.open(sample_paths[0]).convert("RGB"))
    print(
        "Shape (H, W, C):", sample_image.shape,
        "dtype:", sample_image.dtype,
        "range:", int(sample_image.min()), "to", int(sample_image.max()),
    )
    print("Top-left 3x3 of the red channel:\n", sample_image[:3, :3, 0])
else:
    print("Image grid skipped: attach the Intel dataset in Kaggle, then rerun this cell.")

Image grid skipped: attach the Intel dataset in Kaggle, then rerun this cell.


### Part 1: Write It

After the image grid appears, write your own visual observation and record the image facts printed above:

- The two classes that look most similar to me are: **[fill after inspecting the grid]**.
- My predicted most-confused pair is: **[choose before checking Part 6]**.
- Sample image: shape **[H, W, C]**, dtype **[printed dtype]**, pixel range **[printed minimum to maximum]**.

Do not substitute guesses for the measured array values.

In [4]:
# Task 2.1: verify the lecture convolution on a 5x5 vertical edge.
edge_image = torch.tensor([[0, 0, 10, 10, 10]] * 5, dtype=torch.float32)
edge_kernel = torch.tensor([[-1, 0, 1]] * 3, dtype=torch.float32)
edge_output = F.conv2d(
    edge_image.view(1, 1, 5, 5), edge_kernel.view(1, 1, 3, 3)
).squeeze()
expected_output = torch.tensor([[30, 30, 0]] * 3, dtype=torch.float32)
print("Valid convolution output:\n", edge_output)
assert torch.equal(edge_output, expected_output)


def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters())


# Task 2.2: compare local convolutional filters with a dense layer.
conv = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1)
dense = nn.Linear(150 * 150 * 3, 512)
conv_parameters = count_parameters(conv)
dense_parameters = count_parameters(dense)
print(f"Conv 3x3, 3 -> 64: {conv_parameters:,} parameters")
print(f"Dense 67,500 -> 512: {dense_parameters:,} parameters")
print(f"Dense-to-convolution parameter ratio: {dense_parameters // conv_parameters:,}x")
assert conv_parameters == 1_792
assert dense_parameters == 34_560_512

image_tensor = torch.randn(1, 3, 224, 224)
for stride in (1, 2):
    layer = nn.Conv2d(3, 64, kernel_size=3, stride=stride, padding=1)
    print(f"stride {stride}: output {tuple(layer(image_tensor).shape)}")
print("2x2 max pool:", tuple(F.max_pool2d(image_tensor, 2).shape))

if DATA_AVAILABLE:
    gray = transforms.functional.to_tensor(
        Image.fromarray(sample_image).convert("L")
    ).unsqueeze(0)
    edge_map = F.conv2d(
        gray, edge_kernel.view(1, 1, 3, 3), padding=1
    )
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(gray.squeeze(), cmap="gray")
    axes[0].set_title("Input")
    axes[1].imshow(edge_map.squeeze().abs(), cmap="gray")
    axes[1].set_title("Vertical edges")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Photo edge visualization skipped until the dataset is attached.")

Valid convolution output:
 tensor([[30., 30.,  0.],
        [30., 30.,  0.],
        [30., 30.,  0.]])
Conv 3x3, 3 -> 64: 1,792 parameters
Dense 67,500 -> 512: 34,560,512 parameters
Dense-to-convolution parameter ratio: 19,286x
stride 1: output (1, 64, 224, 224)
stride 2: output (1, 64, 112, 112)
2x2 max pool: (1, 3, 112, 112)
Photo edge visualization skipped until the dataset is attached.


### Part 2: Write It

A 3x3 RGB filter with 64 outputs has `(3 x 3 x 3 + 1) x 64 = 1,792` parameters. The dense layer has `(67,500 + 1) x 512 = 34,560,512`, exactly **19,286 times** as many. A convolution reuses each filter at every image location, so a learned edge or texture can activate wherever it appears; this weight sharing both reduces parameters and gives translation equivariance.

In [5]:
# ImageNet normalization expected by the pretrained ResNet-18 weights.
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

train_tf = transforms.Compose(
    [
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)
eval_tf = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)

DATA_LOADERS_READY = False
if DATA_AVAILABLE:
    full_aug = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
    full_eval = datasets.ImageFolder(TRAIN_DIR, transform=eval_tf)
    if full_aug.class_to_idx != full_eval.class_to_idx:
        raise ValueError("Training and validation class mappings differ.")

    indices = np.random.permutation(len(full_aug))
    validation_size = int(0.15 * len(indices))
    train_indices = indices[validation_size:]
    validation_indices = indices[:validation_size]
    train_ds = Subset(full_aug, train_indices)
    val_ds = Subset(full_eval, validation_indices)
    test_ds = datasets.ImageFolder(TEST_DIR, transform=eval_tf)

    worker_count = 0 if os.name == "nt" else 2
    loader_options = {
        "num_workers": worker_count,
        "pin_memory": device.type == "cuda",
    }
    train_dl = DataLoader(
        train_ds, batch_size=64, shuffle=True, **loader_options
    )
    val_dl = DataLoader(
        val_ds, batch_size=128, shuffle=False, **loader_options
    )
    test_dl = DataLoader(
        test_ds, batch_size=128, shuffle=False, **loader_options
    )
    DATA_LOADERS_READY = True
    print(
        f"{len(train_ds)} train | {len(val_ds)} validation | "
        f"{len(test_ds)} test"
    )
else:
    print("Data loaders skipped: attach the dataset and rerun this cell.")

Data loaders skipped: attach the dataset and rerun this cell.


### Part 3: Write It

Record the train, validation, and test counts printed by the loader cell. Validation is used to choose epochs and compare training phases; the test set is held back until Part 6. Vertical flips can make outdoor scenes physically implausible by putting the ground above the sky, while horizontal flips can reverse handwritten digits such as 2, 3, 5, or 7.

In [6]:
def run_epoch(model, loader, optimizer=None):
    """Run one training pass when optimizer is set, otherwise evaluate."""
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    correct = 0
    sample_count = 0

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            correct += (logits.argmax(dim=1) == labels).sum().item()
            sample_count += batch_size

    if sample_count == 0:
        raise ValueError("Cannot evaluate an empty data loader.")
    return total_loss / sample_count, correct / sample_count


def fit(model, optimizer, epochs, history):
    """Train for a phase and restore the best validation-accuracy weights."""
    best_accuracy = -1.0
    best_state = None
    for epoch in range(epochs):
        started = time.time()
        train_loss, train_accuracy = run_epoch(model, train_dl, optimizer)
        validation_loss, validation_accuracy = run_epoch(model, val_dl)
        history.append(
            (train_loss, validation_loss, train_accuracy, validation_accuracy)
        )
        if validation_accuracy > best_accuracy:
            best_accuracy = validation_accuracy
            best_state = copy.deepcopy(model.state_dict())
        print(
            f"epoch {epoch + 1}: train loss {train_loss:.3f} "
            f"acc {train_accuracy:.3f} | val loss {validation_loss:.3f} "
            f"acc {validation_accuracy:.3f} ({time.time() - started:.0f}s)"
        )
    if best_state is None:
        raise RuntimeError("Training did not produce a valid validation checkpoint.")
    model.load_state_dict(best_state)
    return best_accuracy


TRAINING_READY = DATA_LOADERS_READY and device.type == "cuda"
if TRAINING_READY:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, len(classes))
    model = model.to(device)

    trainable_parameters = sum(
        parameter.numel() for parameter in model.parameters()
        if parameter.requires_grad
    )
    print(f"Trainable {trainable_parameters:,} of {count_parameters(model):,} parameters")
    assert trainable_parameters == 3_078

    history = []
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
    best_feature_extraction = fit(model, optimizer, epochs=3, history=history)
    feature_extraction_history = list(history)
    print(
        "Best validation accuracy, feature extraction: "
        f"{best_feature_extraction:.3f}"
    )
else:
    print("Training skipped: Kaggle GPU and dataset are both required.")

Training skipped: Kaggle GPU and dataset are both required.


### Part 4: Write It

Record the three validation accuracies printed for feature extraction. The pretrained ResNet already contains useful low- and mid-level filters for edges, textures, and shapes. Training only the new classifier head lets those reusable features map to the six scene classes, so high accuracy is possible without changing the frozen backbone.

In [7]:
if TRAINING_READY:
    for parameter in model.layer4.parameters():
        parameter.requires_grad = True

    optimizer = torch.optim.Adam(
        [
            {"params": model.layer4.parameters(), "lr": 1e-4},
            {"params": model.fc.parameters(), "lr": 1e-3},
        ]
    )
    trainable_parameters = sum(
        parameter.numel() for parameter in model.parameters()
        if parameter.requires_grad
    )
    print(f"Trainable after unfreezing layer4: {trainable_parameters:,}")
    best_fine_tuning = fit(model, optimizer, epochs=3, history=history)
    fine_tuning_history = list(history[3:])
    print(f"Best validation accuracy, fine-tuned: {best_fine_tuning:.3f}")

    history_array = np.asarray(history)
    epochs = np.arange(1, len(history_array) + 1)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, history_array[:, 0], "o-", label="train")
    axes[0].plot(epochs, history_array[:, 1], "o-", label="validation")
    axes[1].plot(epochs, history_array[:, 2], "o-", label="train")
    axes[1].plot(epochs, history_array[:, 3], "o-", label="validation")
    for axis, title in zip(axes, ("Loss", "Accuracy")):
        axis.axvline(3.5, color="gray", linestyle="--")
        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Fine-tuning and curves skipped: run Part 4 in Kaggle with GPU and data.")

Fine-tuning and curves skipped: run Part 4 in Kaggle with GPU and data.


### Part 5: Write It

Compare the best fine-tuned validation accuracy with feature extraction: **[report the measured difference]**. Describe the training/validation loss and accuracy curves, including any divergence that suggests overfitting. Training accuracy may be lower because each training batch contains random crops, flips, and color changes, while validation uses clean resized images.

In [8]:
TEST_EVALUATED = False
if TRAINING_READY and "best_fine_tuning" in globals():
    model.eval()
    probability_batches = []
    prediction_batches = []
    label_batches = []
    with torch.no_grad():
        for images, labels_batch in test_dl:
            logits = model(images.to(device, non_blocking=True))
            probabilities = F.softmax(logits, dim=1).cpu()
            probability_batches.append(probabilities)
            prediction_batches.append(probabilities.argmax(dim=1))
            label_batches.append(labels_batch.cpu())

    probabilities = torch.cat(probability_batches)
    predictions = torch.cat(prediction_batches)
    labels = torch.cat(label_batches)
    test_accuracy = float((predictions == labels).float().mean())
    print(f"Test accuracy: {test_accuracy:.3f}")
    print(
        classification_report(
            labels.numpy(),
            predictions.numpy(),
            target_names=classes,
            digits=3,
            zero_division=0,
        )
    )

    confusion = confusion_matrix(labels.numpy(), predictions.numpy())
    display_plot = ConfusionMatrixDisplay(
        confusion_matrix=confusion,
        display_labels=classes,
    )
    display_plot.plot(cmap="Blues", xticks_rotation=45)
    plt.tight_layout()
    plt.show()

    misclassified = torch.where(predictions != labels)[0][:8].tolist()
    figure, axes = plt.subplots(2, 4, figsize=(14, 7))
    for axis in axes.flat:
        axis.axis("off")
    for axis, sample_index in zip(axes.flat, misclassified):
        image_path, _ = test_ds.samples[sample_index]
        with Image.open(image_path) as error_image:
            axis.imshow(error_image.convert("RGB"))
        axis.set_title(
            f"true: {classes[labels[sample_index]]}\n"
            f"pred: {classes[predictions[sample_index]]} "
            f"({probabilities[sample_index].max():.2f})"
        )
    if misclassified:
        plt.tight_layout()
        plt.show()
    else:
        print("No misclassified test images were found.")

    sample_image_tensor, true_label = test_ds[0]
    with torch.no_grad():
        sample_logits = model(sample_image_tensor.unsqueeze(0).to(device)).cpu().squeeze(0)
    sample_probabilities = F.softmax(sample_logits, dim=0)
    print("True class:", classes[true_label])
    for class_name, logit, probability in zip(
        classes, sample_logits.tolist(), sample_probabilities.tolist()
    ):
        print(f"{class_name:10s} logit {logit:6.2f}   probability {probability:.3f}")
    TEST_EVALUATED = True
else:
    print("Test evaluation skipped: complete Parts 3-5 on Kaggle first.")

Test evaluation skipped: complete Parts 3-5 on Kaggle first.


### Part 6: Write It

Use the printed test report and confusion matrix; do not tune the model based on these test results.

- Test accuracy: **[measured value]**.
- Two weakest classes by recall: **[class and recall]**, **[class and recall]**.
- Most-confused pair (actual -> predicted): **[classes]**. My Part 1 prediction was **[right/wrong]**.
- Error 1: **[wrong prediction, label issue, or genuinely ambiguous image, with visual evidence]**.
- Error 2: **[wrong prediction, label issue, or genuinely ambiguous image, with visual evidence]**.

Softmax probabilities should be nonnegative and sum to 1; the largest probability is the model's predicted class, not a guarantee of correctness.

In [9]:
OUTPUT_DIR = Path("/kaggle/working")
if TEST_EVALUATED and OUTPUT_DIR.is_dir():
    checkpoint_path = OUTPUT_DIR / "resnet18_intel.pt"
    torch.save(model.state_dict(), checkpoint_path)
    print(
        f"Saved {checkpoint_path} "
        f"({checkpoint_path.stat().st_size / 1e6:.1f} MB)"
    )

    report = classification_report(
        labels.numpy(),
        predictions.numpy(),
        target_names=classes,
        output_dict=True,
        zero_division=0,
    )
    recalls = {
        class_name: report[class_name]["recall"] for class_name in classes
    }
    weakest_classes = sorted(recalls.items(), key=lambda item: item[1])[:2]
    off_diagonal = confusion.copy()
    np.fill_diagonal(off_diagonal, 0)
    if off_diagonal.max() > 0:
        actual_index, predicted_index = np.unravel_index(
            off_diagonal.argmax(), off_diagonal.shape
        )
        confused_pair = (
            f"{classes[actual_index]} -> {classes[predicted_index]} "
            f"({off_diagonal[actual_index, predicted_index]} images)"
        )
    else:
        confused_pair = "No off-diagonal errors"

    print("Weakest recalls:", weakest_classes)
    print("Most confused pair:", confused_pair)
    print(f"Feature-extraction best val accuracy: {best_feature_extraction:.3f}")
    print(f"Fine-tuning best val accuracy: {best_fine_tuning:.3f}")

    readme_text = f"""# Project 2, Week 5: Scene Classification with Transfer Learning

- Dataset: Intel Image Classification, 6 classes, {len(train_ds)} train / {len(val_ds)} validation / {len(test_ds)} test images
- Model: ResNet-18 pretrained on ImageNet
- Feature extraction (3,078 trainable parameters): best validation accuracy {best_feature_extraction:.3f}
- Fine-tuning layer4 (about 8.4M trainable parameters): best validation accuracy {best_fine_tuning:.3f}
- Test accuracy (used once): {test_accuracy:.3f}
- Two weakest classes by recall: {weakest_classes[0][0]} ({weakest_classes[0][1]:.3f}), {weakest_classes[1][0]} ({weakest_classes[1][1]:.3f})
- Most confused pair (actual -> predicted): {confused_pair}
- Most instructive error: [Describe one image after reviewing the error gallery.]
- Next, my Week 6 domain: [State your domain, who benefits, candidate dataset URL, classes, and image count.]

## Reproduce

Run the notebook in Kaggle with a T4 GPU and Internet enabled. The checkpoint is `resnet18_intel.pt`.
"""
    readme_path = OUTPUT_DIR / "README.md"
    readme_path.write_text(readme_text, encoding="utf-8")
    print(f"Saved README draft: {readme_path}")

    linkedin_text = f"""Week 5 of Applied AI: my first deep learning model.

I trained a ResNet-18 to recognize six kinds of scenes with {test_accuracy:.1%} test accuracy.

Transfer learning let me start with ImageNet features, train a 3,078-parameter head, then fine-tune the last block.

Its largest confusion was {confused_pair}. [Add one honest observation from the error gallery.]

#DeepLearning #PyTorch #ComputerVision #LearningInPublic
"""
    print("\nLinkedIn draft (add your error screenshot and observation):\n")
    print(linkedin_text)
elif TEST_EVALUATED:
    print("Test results exist, but /kaggle/working is unavailable in this runtime.")
else:
    print("Saving skipped: finish Parts 3-6 on Kaggle and evaluate the test set once.")

Saving skipped: finish Parts 3-6 on Kaggle and evaluate the test set once.


### Part 7: Write It and Check Your Understanding

The Kaggle run writes `resnet18_intel.pt` and a README draft containing measured results. Review the error gallery and replace the remaining bracketed observations with your own. For the Week 6 proposal, name the problem and beneficiary, link a licensed candidate dataset, and report its classes and image counts. Add the generated real test accuracy and one honest error to your LinkedIn post; do not post the draft without completing its observation.

#### Check Your Understanding

1. A composition of linear layers (or affine layers with biases) is still one linear/affine transformation. Without a nonlinear activation, depth cannot represent a more complex nonlinear decision boundary.
2. For logits `[3, 1, 0]`, the first softmax probability is `e^3 / (e^3 + e^1 + e^0) = 20.086 / (20.086 + 2.718 + 1) ~= 0.844`.
3. A 3x3 convolution from 64 to 128 channels has `(3 x 3 x 64 + 1) x 128 = 73,856` parameters, including one bias per output channel.
4. With input 56, kernel 3, padding 1, and stride 2: `floor((56 + 2 - 3) / 2) + 1 = 28`, so the output is `28 x 28`.
5. A residual connection adds an identity path around the block, allowing gradients to flow directly to earlier layers instead of passing only through every transformation.
6. Training from scratch is reasonable when there is enough labeled data and compute, when the target domain differs substantially from ImageNet, or when the goal requires learning domain-specific low-level features.